In [8]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.special import betaln
from itertools import combinations
import yaml
import matplotlib.patches as mpatches
import yaml

with open("cfg.yml", "r") as f:
    cfg = yaml.safe_load(f)
with open("cfg.yml", "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f) or {}

for key, value in CFG.items():
    globals()[key] = value

NET_SIZES = cfg["net_size"]                         # 10, 25, 50, 100, 200
TOPOLOGIES = cfg["topology"]                        # "cycle", "wheel", ["clustered", "scale_free", "random"]
THEORIES = cfg["theories"]                          # [0.82, 0.81, 0.80, 0.79, 0.78], [0.9, 0.85, 0.8, 0.75, 0.7], [0.82, 0.81, 0.80, 0.70, 0.50] 
TOP_PRIORS_STRENGTH = cfg["top_priors_strength"]    # 4, 40, 400, 4000
DENSITIES = cfg["density"]                          # 0 to 1, step 0.1
STEPS = 100
N_RUNS = 10 
SEED = 1

RNG = np.random.default_rng(SEED)

In [9]:
# ─────────────────────────────────────────────
# 1. GRAPH FACTORIES
# ─────────────────────────────────────────────

import networkx as nx

def add_semi_directed_hub(G: nx.Graph, outward: bool = True) -> tuple[nx.DiGraph, int]:
    """
    Convert G to a directed graph where:
    - original edges become bidirectional (u ↔ v)
    - hub edges are one-way only

    outward:
        True  → hub → nodes
        False → nodes → hub
    """
    DG = nx.DiGraph()

    # Add original nodes
    DG.add_nodes_from(G.nodes())

    # Make all original edges bidirectional
    for u, v in G.edges():
        DG.add_edge(u, v)
        DG.add_edge(v, u)

    # Add hub
    hub = max(G.nodes) + 1
    DG.add_node(hub)

    for node in G.nodes():
        if outward:
            DG.add_edge(hub, node)   # hub → node
        else:
            DG.add_edge(node, hub)   # node → hub

    return DG, hub

def make_graphs(ns: list[int], densities: list[float], outward: bool = True) -> dict[int, dict[str, nx.DiGraph | dict[float, nx.DiGraph]]]:
    graphs_by_n = {}

    for n in ns:
        entry = {
            "Wheel": None,
            "Cycle": None,
            "Random":      {},
            "Scale-Free":  {},
            "Clustered":   {},
        }

        # Fixed-topology graphs
        entry["Wheel"], _ = add_semi_directed_hub(nx.wheel_graph(n), outward=outward)
        entry["Cycle"], _ = add_semi_directed_hub(nx.cycle_graph(n), outward=outward)

        for density in densities:
            p = density
            m = max(1, int(density * (n - 1)))
            k = min(max(2, int(density * n)), n - 2)
            if k % 2 == 1:
                k += 1

            entry["Random"][density],     _ = add_semi_directed_hub(nx.erdos_renyi_graph(n, p),          outward=outward)
            entry["Scale-Free"][density], _ = add_semi_directed_hub(nx.barabasi_albert_graph(n, m),      outward=outward)
            entry["Clustered"][density],  _ = add_semi_directed_hub(nx.watts_strogatz_graph(n, k, p),    outward=outward)

        graphs_by_n[n] = entry

    return graphs_by_n

In [10]:
import matplotlib.pyplot as plt
import networkx as nx

graphs = make_graphs(NET_SIZES, DENSITIES)

# colors = {
#     "Wheel":      "#7F77DD",
#     "Cycle":      "#1D9E75",
#     "Random":     "#3A86FF",
#     "Scale-Free": "#FF006E",
#     "Clustered":  "#8338EC",
# }

# for n, density_graphs in graphs.items():
#     for density in DENSITIES:
#         fig, axes = plt.subplots(1, len(density_graphs), figsize=(5 * len(density_graphs), 4))
#         axes = axes.ravel() if hasattr(axes, "ravel") else [axes]

#         for ax, (name, G_or_dict) in zip(axes, density_graphs.items()):
#             G = G_or_dict if isinstance(G_or_dict, nx.Graph) else G_or_dict[density]
#             hub = max(G.nodes())

#             # --- Layouts ---
#             if name == "Wheel":
#                 pos = nx.shell_layout(G, nlist=[list(range(n)), [hub]])
#             elif name in ["Scale-Free", "Random"]:
#                 pos = nx.spring_layout(G, seed=SEED)  # spring handles hub naturally
#             else:
#                 pos = nx.circular_layout(G)
#                 # Place hub in the center for non-spring layouts
#                 pos[hub] = np.array([0.0, 0.0])

#             node_colors = [
#                             "#FF0000" if v == hub else
#                             "#0F6E56" if (name == "Wheel" and v == 0) else
#                             colors[name]
#                             for v in G.nodes()
#                         ]

#             edge_colors = [
#                 "#FF0000" if (u == hub or v == hub) else "#B4B2A9"
#                 for u, v in G.edges()
#             ]

#             # Split edges
#             bidirectional = [(u, v) for u, v in G.edges() if G.has_edge(v, u) and u != hub and v != hub]
#             hub_edges     = [(u, v) for u, v in G.edges() if u == hub or v == hub]

#             # Draw nodes
#             nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, node_size=200 if n > 100 else 500)
#             nx.draw_networkx_labels(G, pos, ax=ax, font_color="white", font_size=6 if n > 100 else 10, font_weight="bold")

#             # Bidirectional edges — no arrows
#             nx.draw_networkx_edges(G, pos, ax=ax, edgelist=bidirectional, edge_color="#B4B2A9", arrows=False, width=1.0)

#             # Hub edges — arrows
#             nx.draw_networkx_edges(G, pos, ax=ax, edgelist=hub_edges, edge_color="#FF0000", arrows=True, arrowsize=8, connectionstyle="arc3,rad=0.08", width=1.0)


#             ax.set_title(
#                 f"{name}  |  {G.number_of_nodes()}V  {G.number_of_edges()}E",
#                 fontsize=10,
#                 pad=10,
#             )
#             ax.axis("off")

#         plt.suptitle(f"n={n}  |  Density = {density:.1f}", fontsize=12)
#         plt.tight_layout(rect=[0, 0, 1, 0.95])
#         plt.show()

In [11]:
G = graphs[25]['Random'][0.5]

print(f"Nodes:         {G.number_of_nodes()}")
print(f"Edges:         {G.number_of_edges()}")
print(f"Directed:      {G.is_directed()}")
print(f"Density:       {nx.density(G):.3f}")
print(f"Avg degree:    {sum(d for _, d in G.degree()) / G.number_of_nodes():.2f}")

Nodes:         26
Edges:         327
Directed:      True
Density:       0.503
Avg degree:    25.15


In [12]:
# ─────────────────────────────────────────────
# 2. NODE PRIORS INITIALIZATION
# ─────────────────────────────────────────────

def assign_node_values(G: nx.DiGraph, top_prior: float, hub: int) -> dict:
    values = {}
    for node in G.nodes():
        if node == hub:
            values[node] = 0
        else:
            alpha = RNG.uniform(0, top_prior)
            beta  = RNG.uniform(0, top_prior)
            values[node] = [alpha, beta]
    return values

for prior in TOP_PRIORS_STRENGTH:
    for n, n_graphs in graphs.items():
        for name, G_or_dict in n_graphs.items():
            g_items = [(None, G_or_dict)] if isinstance(G_or_dict, nx.DiGraph) else G_or_dict.items()

            for density, G in g_items:
                hub = max(G.nodes())

                for node in G.nodes():
                    if prior not in G.nodes[node]:
                        G.nodes[node][prior] = {}

                for theory_set in THEORIES:
                    theory_set_key = tuple(theory_set)  # use tuple as dict key

                    for node in G.nodes():
                        G.nodes[node][prior][theory_set_key] = {}

                    for theory in theory_set:
                        node_values = assign_node_values(G, prior, hub)

                        EV = {
                            node: val[0] / (val[0] + val[1])
                            for node, val in node_values.items()
                            if node != hub
                        }

                        for node in G.nodes():
                            G.nodes[node][prior][theory_set_key][theory] = {
                                "ab": node_values[node],
                                "EV": EV.get(node, None),
                            }

In [13]:
# Inspect

G = graphs[10]["Random"][0.5]  # or graphs[5]["Random"][0.1] for density-varying
node = 2
theories_1 = (0.82, 0.81, 0.80, 0.70, 0.50)
theories_2 = (0.9, 0.85, 0.8, 0.75, 0.7)
theories_3 = (0.82, 0.81, 0.8, 0.79, 0.78)

print(G.nodes[node][40][theories_3][0.78])

{'ab': [38.3022735219263, 31.584461076467534], 'EV': 0.5480621428663313}


In [14]:
graphs_1 = graphs

import pickle

with open("graphs_1.pkl", "wb") as f:
    pickle.dump(graphs_1, f)
